# Assignment 3: Image Captioning in Google Colab

This is a single self-contained notebook for the final paper implementation.

It includes:
- dataset loading from Hugging Face `yerevann/coco-karpathy`
- `ResNet-50 + LSTM` baseline
- `ResNet-50 + Transformer` proposed model
- training and validation loops
- BLEU and METEOR evaluation
- qualitative sample generation

You do not need any local Python files for this workflow.

In [ ]:
!pip install -q torch torchvision datasets requests tqdm nltk pillow matplotlib

In [ ]:
import io
import json
import os
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Sequence

import requests
import torch
import torch.nn as nn
from datasets import load_dataset
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

REQUIRE_CUDA_FOR_TRAINING = True
if REQUIRE_CUDA_FOR_TRAINING and not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is not available. In Colab go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, then restart the runtime and re-run the notebook from the top.'
    )

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_SIZE = 10000
VAL_SIZE = 1000
TEST_SIZE = 1000
BATCH_SIZE = 16
EPOCHS = 10
MAX_LEN = 30
MIN_WORD_FREQ = 5
CACHE_DIR = '/content/coco_cache'
OUTPUT_DIR = '/content/assignment3_outputs'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOKEN_PATTERN = re.compile(r"[a-z0-9']+")

def tokenize_caption(text: str) -> List[str]:
    return TOKEN_PATTERN.findall(text.lower())

@dataclass
class Vocabulary:
    stoi: Dict[str, int]
    itos: List[str]
    
    @classmethod
    def build(cls, captions: Sequence[str], min_freq: int = 5):
        specials = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
        counter = Counter()
        for caption in captions:
            counter.update(tokenize_caption(caption))
        tokens = specials[:]
        for word, freq in counter.items():
            if freq >= min_freq:
                tokens.append(word)
        stoi = {token: idx for idx, token in enumerate(tokens)}
        return cls(stoi=stoi, itos=tokens)
    
    @property
    def pad_idx(self):
        return self.stoi['<PAD>']
    
    @property
    def sos_idx(self):
        return self.stoi['<SOS>']
    
    @property
    def eos_idx(self):
        return self.stoi['<EOS>']
    
    @property
    def unk_idx(self):
        return self.stoi['<UNK>']
    
    def __len__(self):
        return len(self.itos)
    
    def encode(self, caption: str, max_length: int):
        tokens = tokenize_caption(caption)
        ids = [self.sos_idx]
        ids.extend(self.stoi.get(token, self.unk_idx) for token in tokens[: max_length - 2])
        ids.append(self.eos_idx)
        return torch.tensor(ids, dtype=torch.long)
    
    def decode(self, token_ids: Sequence[int]):
        words = []
        for token_id in token_ids:
            if token_id == self.eos_idx:
                break
            if token_id in (self.pad_idx, self.sos_idx):
                continue
            if 0 <= token_id < len(self.itos):
                words.append(self.itos[token_id])
        return ' '.join(words)

def build_vocab(train_limit=TRAIN_SIZE):
    ds = load_dataset('yerevann/coco-karpathy', split='train')
    ds = ds.shuffle(seed=SEED).select(range(min(train_limit, len(ds))))
    captions = []
    for item in ds:
        captions.extend(item['sentences'])
    return Vocabulary.build(captions, min_freq=MIN_WORD_FREQ)

vocab = build_vocab()
print('Vocab size:', len(vocab))

class CocoKarpathyDataset(Dataset):
    def __init__(self, split, vocab, limit, train_mode=True):
        self.vocab = vocab
        self.train_mode = train_mode
        self.records = load_dataset('yerevann/coco-karpathy', split=split)
        self.records = self.records.shuffle(seed=SEED).select(range(min(limit, len(self.records))))
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        if train_mode:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                normalize,
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                normalize,
            ])
    
    def __len__(self):
        return len(self.records)
    
    def _fetch_image(self, url, filename):
        cache_path = os.path.join(CACHE_DIR, filename)
        if os.path.exists(cache_path):
            return Image.open(cache_path).convert('RGB')
        for _ in range(3):
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                image = Image.open(io.BytesIO(response.content)).convert('RGB')
                image.save(cache_path)
                return image
            except Exception:
                pass
        return Image.new('RGB', (224, 224), color=(255, 255, 255))
    
    def __getitem__(self, index):
        item = self.records[index]
        image = self._fetch_image(item['url'], item['filename'])
        image_tensor = self.transform(image)
        if self.train_mode:
            caption = random.choice(item['sentences'])
            return {
                'image': image_tensor,
                'caption_ids': self.vocab.encode(caption, MAX_LEN)
            }
        return {
            'image': image_tensor,
            'references': [caption.strip().lower() for caption in item['sentences']],
            'image_id': item['cocoid'],
            'filename': item['filename']
        }

def train_collate_fn(batch):
    images = torch.stack([item['image'] for item in batch], dim=0)
    captions = pad_sequence([item['caption_ids'] for item in batch], batch_first=True, padding_value=vocab.pad_idx)
    return {'images': images, 'captions': captions}

def eval_collate_fn(batch):
    return {
        'images': torch.stack([item['image'] for item in batch], dim=0),
        'references': [item['references'] for item in batch],
        'image_ids': [item['image_id'] for item in batch],
        'filenames': [item['filename'] for item in batch]
    }

class ResNetEncoder(nn.Module):
    def __init__(self, trainable=False):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.output_dim = 2048
        if not trainable:
            for p in self.backbone.parameters():
                p.requires_grad = False
    def forward(self, images):
        return self.backbone(images).flatten(start_dim=1)

class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, feature_dim=512, embedding_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)
    def forward(self, features, captions):
        embeddings = self.embedding(captions)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        outputs, _ = self.lstm(embeddings, (h0, c0))
        return self.output(outputs)
    def generate(self, features, start_idx, max_length):
        batch_size = features.size(0)
        h = self.init_h(features).unsqueeze(0)
        c = self.init_c(features).unsqueeze(0)
        current = torch.full((batch_size,), start_idx, dtype=torch.long, device=features.device)
        generated = []
        for _ in range(max_length):
            emb = self.embedding(current).unsqueeze(1)
            outputs, (h, c) = self.lstm(emb, (h, c))
            logits = self.output(outputs.squeeze(1))
            current = logits.argmax(dim=-1)
            generated.append(current)
        return torch.stack(generated, dim=1)

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, model_dim=512, num_heads=8, num_layers=4, ff_dim=2048, dropout=0.1, max_length=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.position_embedding = nn.Embedding(max_length, model_dim)
        layer = nn.TransformerDecoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.output = nn.Linear(model_dim, vocab_size)
    def _mask(self, seq_len, device):
        return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    def forward(self, memory, captions):
        positions = torch.arange(captions.size(1), device=captions.device).unsqueeze(0)
        tokens = self.token_embedding(captions) + self.position_embedding(positions)
        decoded = self.decoder(tokens, memory.unsqueeze(1), tgt_mask=self._mask(captions.size(1), captions.device))
        return self.output(decoded)
    def generate(self, memory, start_idx, max_length):
        generated = torch.full((memory.size(0), 1), start_idx, dtype=torch.long, device=memory.device)
        for _ in range(max_length):
            logits = self.forward(memory, generated)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
        return generated[:, 1:]

class BaselineCaptionModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = ResNetEncoder(trainable=False)
        self.project = nn.Linear(self.encoder.output_dim, 512)
        self.decoder = LSTMDecoder(vocab_size=vocab_size, feature_dim=512)
    def forward(self, images, captions):
        features = self.project(self.encoder(images))
        return self.decoder(features, captions)
    def generate(self, images, start_idx, max_length):
        features = self.project(self.encoder(images))
        return self.decoder.generate(features, start_idx, max_length)

class TransformerCaptionModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = ResNetEncoder(trainable=False)
        self.project = nn.Linear(self.encoder.output_dim, 512)
        self.decoder = TransformerDecoder(vocab_size=vocab_size, model_dim=512, num_heads=8, num_layers=4, ff_dim=2048, dropout=0.1, max_length=MAX_LEN + 5)
    def forward(self, images, captions):
        memory = self.project(self.encoder(images))
        return self.decoder(memory, captions)
    def generate(self, images, start_idx, max_length):
        memory = self.project(self.encoder(images))
        return self.decoder.generate(memory, start_idx, max_length)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_one_epoch(model, loader, optimizer, criterion, desc='train'):
    model.train()
    total = 0.0
    progress = tqdm(loader, desc=desc, leave=False)
    for batch in progress:
        images = batch['images'].to(device)
        captions = batch['captions'].to(device)
        inputs = captions[:, :-1]
        targets = captions[:, 1:]
        optimizer.zero_grad(set_to_none=True)
        logits = model(images, inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss.backward()
        optimizer.step()
        total += loss.item()
        progress.set_postfix(loss=f'{loss.item():.4f}')
    return total / max(len(loader), 1)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, desc='val'):
    model.eval()
    total = 0.0
    progress = tqdm(loader, desc=desc, leave=False)
    for batch in progress:
        images = batch['images'].to(device)
        captions = batch['captions'].to(device)
        inputs = captions[:, :-1]
        targets = captions[:, 1:]
        logits = model(images, inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        total += loss.item()
        progress.set_postfix(loss=f'{loss.item():.4f}')
    return total / max(len(loader), 1)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    predictions = []
    references = []
    records = []
    for batch in loader:
        images = batch['images'].to(device)
        token_ids = model.generate(images, vocab.sos_idx, MAX_LEN)
        decoded = [vocab.decode(tokens.tolist()) for tokens in token_ids.cpu()]
        predictions.extend(decoded)
        references.extend(batch['references'])
        for image_id, filename, refs, pred in zip(batch['image_ids'], batch['filenames'], batch['references'], decoded):
            records.append({'image_id': image_id, 'filename': filename, 'references': refs, 'prediction': pred})
    smooth = SmoothingFunction().method1
    bleu_refs = [[ref.split() for ref in refs] for refs in references]
    bleu_preds = [pred.split() for pred in predictions]
    bleu1 = corpus_bleu(bleu_refs, bleu_preds, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smooth) * 100.0
    bleu4 = corpus_bleu(bleu_refs, bleu_preds, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth) * 100.0
    meteor = sum(meteor_score([ref.split() for ref in refs], pred.split()) for refs, pred in zip(references, predictions)) / max(len(predictions), 1) * 100.0
    return {'BLEU-1': bleu1, 'BLEU-4': bleu4, 'METEOR': meteor}, records

train_ds = CocoKarpathyDataset('train', vocab, TRAIN_SIZE, train_mode=True)
val_ds = CocoKarpathyDataset('validation', vocab, VAL_SIZE, train_mode=True)
test_ds = CocoKarpathyDataset('test', vocab, TEST_SIZE, train_mode=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=train_collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=train_collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=eval_collate_fn)

print('Train/Val/Test sizes:', len(train_ds), len(val_ds), len(test_ds))

CUDA available: False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/246 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/82783 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating restval split:   0%|          | 0/30504 [00:00<?, ? examples/s]

Vocab size: 3338
Train/Val/Test sizes: 10000 1000 1000


In [ ]:
baseline = BaselineCaptionModel(vocab_size=len(vocab)).to(device)
transformer = TransformerCaptionModel(vocab_size=len(vocab)).to(device)
print('Baseline trainable params:', count_params(baseline))
print('Transformer trainable params:', count_params(transformer))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 122MB/s]


Baseline trainable params: 5718282
Transformer trainable params: 21304586


## Train baseline model

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)
optimizer = torch.optim.AdamW(baseline.parameters(), lr=1e-4, weight_decay=1e-5)
best_val = float('inf')
baseline_ckpt = os.path.join(OUTPUT_DIR, 'baseline_best.pt')
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    train_loss = train_one_epoch(baseline, train_loader, optimizer, criterion, desc=f'baseline train {epoch:02d}')
    val_loss = validate_one_epoch(baseline, val_loader, criterion, desc=f'baseline val {epoch:02d}')
    epoch_minutes = (time.time() - epoch_start) / 60.0
    print(f'Baseline epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | time={epoch_minutes:.1f} min')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': baseline.state_dict(), 'vocab_itos': vocab.itos}, baseline_ckpt)
        print('Saved baseline checkpoint:', baseline_ckpt)

Baseline epoch 01 | train_loss=4.9753 | val_loss=4.4123
Saved baseline checkpoint: /content/assignment3_outputs/baseline_best.pt


## Train transformer model

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)
optimizer = torch.optim.AdamW(transformer.parameters(), lr=1e-4, weight_decay=1e-5)
best_val = float('inf')
transformer_ckpt = os.path.join(OUTPUT_DIR, 'transformer_best.pt')
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    train_loss = train_one_epoch(transformer, train_loader, optimizer, criterion, desc=f'transformer train {epoch:02d}')
    val_loss = validate_one_epoch(transformer, val_loader, criterion, desc=f'transformer val {epoch:02d}')
    epoch_minutes = (time.time() - epoch_start) / 60.0
    print(f'Transformer epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | time={epoch_minutes:.1f} min')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': transformer.state_dict(), 'vocab_itos': vocab.itos}, transformer_ckpt)
        print('Saved transformer checkpoint:', transformer_ckpt)

## Evaluate baseline

In [ ]:
baseline = BaselineCaptionModel(vocab_size=len(vocab)).to(device)
baseline.load_state_dict(torch.load(baseline_ckpt, map_location=device)['model_state_dict'])
baseline_metrics, baseline_records = evaluate_model(baseline, test_loader)
print('Baseline metrics:', baseline_metrics)
with open(os.path.join(OUTPUT_DIR, 'baseline_predictions.json'), 'w') as f:
    json.dump(baseline_records, f, indent=2)

## Evaluate transformer

In [ ]:
transformer = TransformerCaptionModel(vocab_size=len(vocab)).to(device)
transformer.load_state_dict(torch.load(transformer_ckpt, map_location=device)['model_state_dict'])
transformer_metrics, transformer_records = evaluate_model(transformer, test_loader)
print('Transformer metrics:', transformer_metrics)
with open(os.path.join(OUTPUT_DIR, 'transformer_predictions.json'), 'w') as f:
    json.dump(transformer_records, f, indent=2)

## Generate qualitative samples

In [ ]:
sample_count = 5
samples = random.sample(transformer_records, min(sample_count, len(transformer_records)))
sample_path = os.path.join(OUTPUT_DIR, 'transformer_samples.md')
lines = ['# Qualitative Sample Predictions', '']
for i, sample in enumerate(samples, start=1):
    lines.append(f'## Sample {i}')
    lines.append(f"- Image ID: `{sample['image_id']}`")
    lines.append(f"- File: `{sample['filename']}`")
    lines.append(f"- Prediction: {sample['prediction']}")
    for ref_idx, ref in enumerate(sample['references'], start=1):
        lines.append(f'- Reference {ref_idx}: {ref}')
    lines.append('')
with open(sample_path, 'w') as f:
    f.write('\n'.join(lines))
print('Saved:', sample_path)
print('\n'.join(lines[:30]))

## Save outputs to Google Drive if mounted

In [ ]:
# Uncomment if you mounted Google Drive earlier.
# !mkdir -p '/content/drive/MyDrive/Assignment3_ImageCaptioning'
# !cp -r {OUTPUT_DIR} '/content/drive/MyDrive/Assignment3_ImageCaptioning/'

## What to send back for the paper

After the notebook finishes, send back:
- printed baseline metrics
- printed transformer metrics
- printed parameter counts
- the GPU name
- the contents of `transformer_samples.md`

That is enough to finalize the IEEE paper with real results.